In [1]:
import tarfile
with tarfile.open('aclImdb_v1.tar.gz', 'r:gz') as tar:
    tar.extractall()

/var/folders/zt/f_lrs0nx2jn33c6pmdxrlwpc0000gn/T/ipykernel_5573/219513043.py:3: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


In [2]:
import sys
import os
import pyprind
import pandas as pd
import numpy as np
basepath = 'aclImdb'
labels = {'pos': 1, 'neg': 0}
pbar = pyprind.ProgBar(50000, stream=sys.stdout)

rows = []
for s in ('test', 'train'):
    for l in ('pos', 'neg'):
        path = os.path.join(basepath, s, l)
        for file in sorted(os.listdir(path)):
            with open(os.path.join(path, file), 'r', encoding='utf-8') as infile:
                txt = infile.read()
            rows.append([txt, labels[l]])
            pbar.update()

df = pd.DataFrame(rows, columns=['review', 'sentiment'])

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:02


In [3]:
np.random.seed(0)
df = df.reindex(np.random.permutation(df.index))
df.to_csv('movie_data.csv', index=False, encoding='utf-8')

In [4]:
df = pd.read_csv('movie_data.csv', encoding='utf-8')
# the following column renaming is necessary on some computers:
df = df.rename(columns={"0": "review", "1": "sentiment"})
df.head(3)

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0


In [5]:
from sklearn.feature_extraction.text import CountVectorizer
count = CountVectorizer()

docs = np.array([
    'The sun is shining',
    'The weather is sweet',
    'The sun is shining, the weather is sweet',
    'and one and one is two'
])

bag = count.fit_transform(docs)
print(count.vocabulary_)

{'the': 6, 'sun': 4, 'is': 1, 'shining': 3, 'weather': 8, 'sweet': 5, 'and': 0, 'one': 2, 'two': 7}


In [6]:
print(bag.toarray())

[[0 1 0 1 1 0 1 0 0]
 [0 1 0 0 0 1 1 0 1]
 [0 2 0 1 1 1 2 0 1]
 [2 1 2 0 0 0 0 1 0]]


In [7]:
count_2gram = CountVectorizer(ngram_range=(2, 2))
bag_2gram = count_2gram.fit_transform(docs)

print(count_2gram.vocabulary_)

{'the sun': 8, 'sun is': 7, 'is shining': 1, 'the weather': 9, 'weather is': 10, 'is sweet': 2, 'shining the': 6, 'and one': 0, 'one and': 4, 'one is': 5, 'is two': 3}


In [8]:
print(bag_2gram.toarray())

[[0 1 0 0 0 0 0 1 1 0 0]
 [0 0 1 0 0 0 0 0 0 1 1]
 [0 1 1 0 0 0 1 1 1 1 1]
 [2 0 0 1 1 1 0 0 0 0 0]]


## Inverse Document Frequency (IDF)

A common log-based formulation is:
$$
\mathrm{idf}(t) = \log\left(\frac{N}{\mathrm{df}(t) + 1}\right)
$$

where:
- $N$ is the total number of documents
- $\mathrm{df}(t)$ is the number of documents that contain term $t$

### Why use $+1$ in the denominator?
- The $+1$ is a smoothing term that prevents division by zero when a term has $\mathrm{df}(t)=0$.
- It keeps the expression numerically stable and avoids undefined values.

### Why use the logarithm?
- Raw inverse frequency can grow too quickly and overemphasize extremely rare terms.
- The log compresses large values so low-frequency terms are still upweighted, but not excessively.

In [9]:
from sklearn.feature_extraction.text import TfidfTransformer
count = CountVectorizer()
tfidf = TfidfTransformer(use_idf=True,
                         norm='l2',
                         smooth_idf=True) # +1 is in the notation above
np.set_printoptions(precision=2)

print(tfidf.fit_transform(count.fit_transform(docs)).toarray())



[[0.   0.38 0.   0.57 0.57 0.   0.46 0.   0.  ]
 [0.   0.38 0.   0.   0.   0.57 0.46 0.   0.57]
 [0.   0.46 0.   0.35 0.35 0.35 0.56 0.   0.35]
 [0.66 0.17 0.66 0.   0.   0.   0.   0.33 0.  ]]


In [10]:
df.loc[0, 'review' ][-50:]

'is seven.<br /><br />Title (Brazil): Not Available'

In [11]:
import re
def preprocessor(text):
    text = re.sub(r'<[^>]*>', '', text)
    emoticons = re.findall(r'(?::|;|=)(?:-)?(?:\)|\(|D|P)',
            text)
    text = (re.sub(r'[\W]+', ' ', text.lower()) +
    ' '.join(emoticons).replace('-', ''))
    return text

In [12]:
df['review'] = df['review'].apply(preprocessor)

In [13]:
df.head()

,review,sentiment
0,in 1974 the teenager martha moxley maggie grac...,1
1,ok so i really like kris kristofferson and his...,0
2,spoiler do not read this if you think about w...,0
3,hi for all the people who have seen this wonde...,1
4,i recently bought the dvd forgetting just how ...,0


In [14]:
def tokeninizer(text):
    return text.split()

In [15]:
from nltk.stem.porter import PorterStemmer
porter = PorterStemmer()


def tokenizer_porter(text):
    return [porter.stem(word) for word in text.split()]

tokenizer_porter('runners like running and thus they run')

['runner', 'like', 'run', 'and', 'thu', 'they', 'run']

## Stemming vs Lemmatization

- **Stemming** is a heuristic process that chops word endings to reduce words to a base-like form (the **stem**).
  - It is fast, but the output may be a non-dictionary form (e.g., `running` → `run`, sometimes `studies` → `studi`).

- **Lemmatization** reduces words to their canonical dictionary form (the **lemma**) using vocabulary and often part-of-speech context.
  - It is usually more linguistically accurate than stemming (e.g., `better` → `good` when context/POS is considered), but can be slower.

In practice, stemming is common when speed is more important; lemmatization is preferred when semantic correctness matters more.

In [16]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/blaiseforgwa/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [17]:
from nltk.corpus import stopwords
stop = stopwords.words('english')
[w for w in tokenizer_porter('runners like running and thus they run')
 if w not in stop
 ]


['runner', 'like', 'run', 'thu', 'run']

In [18]:
X_train = df.loc[:25000, 'review'].astype(str).to_numpy()
y_train = df.loc[:25000, 'sentiment'].to_numpy()

X_test = df.loc[25000:, 'review'].astype(str).to_numpy()
y_test = df.loc[25000:, 'sentiment'].to_numpy()

In [19]:
import os
import warnings
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')

tfidf = TfidfVectorizer(
    strip_accents=None,
    lowercase=True,
    preprocessor=None
)

small_param_grid = [
    {
        'vect__ngram_range': [(1, 1)],
        'vect__stop_words': [None],
        'vect__tokenizer': [tokeninizer, tokenizer_porter],
        'clf__l1_ratio': [0],
        'clf__C': [1.0, 10.0],
    },
    {
        'vect__ngram_range': [(1, 1)],
        'vect__stop_words': [stop, None],
        'vect__tokenizer': [tokeninizer],
        'vect__use_idf': [False],
        'vect__norm': [None],
        'clf__l1_ratio': [0],
        'clf__C': [1.0, 10.0],
    },
]

lr_tfidf = Pipeline([
    ('vect', tfidf),
    ('clf', LogisticRegression(solver='liblinear'))
])

gs_lr_tfidf = GridSearchCV(
    lr_tfidf,
    small_param_grid,
    scoring='accuracy',
    cv=5,
    verbose=0,
    n_jobs=1
)

In [20]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    gs_lr_tfidf.fit(X_train, y_train)

In [21]:
print(f'Best parameter set: {gs_lr_tfidf.best_params_}')

Best parameter set: {'clf__C': 10.0, 'clf__l1_ratio': 0, 'vect__ngram_range': (1, 1), 'vect__stop_words': None, 'vect__tokenizer': <function tokeninizer at 0x11c147b00>}


In [22]:
print(f'CV accuracy: {gs_lr_tfidf.best_score_:.3f}')
clf = gs_lr_tfidf.best_estimator_

print(f'Test accuracy: {clf.score(X_test, y_test):.3f}')

CV accuracy: 0.897
Test accuracy: 0.899
